In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline\nimport pytorch_lightning as pl\nfrom pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint\nimport os\n

In [ ]:
from typing import Union

In [ ]:
from PIL import Image as PImage
from PIL.Image import Image

In [ ]:
from pathlib import Path

In [ ]:
from collections import OrderedDict

In [ ]:
import numpy as np

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn, Tensor
from torchvision import (transforms, datasets)
# from torchvision import prototype as P

## Image search

In [ ]:
from typing import Union
from pathlib import Path
from tqdm import tqdm

In [ ]:
import cv2

In [ ]:
from torch import no_grad
from torch.jit import ScriptModule
from torchvision.models import (resnet34, resnet50, wide_resnet50_2)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
size = 256
imsz = 224
IMG_SUFF = {'.jpg', '.jpeg', '.png'}
path = Path('data')

In [ ]:
class ToPILImage(object):
    """Convert inout image to PIL image"""

    def __init__(self, mode=None):
        super().__init__()
        self.to_pil = transforms.ToPILImage(mode=mode)

    def convert(self, img: Union[np.ndarray, Image]) -> Image:
        """
        Converts image to the PIL format
        Args:
            img: inout image

        Returns:
            converted image
        """
        return img if isinstance(img, Image) else self.to_pil(img)

    def __call__(self, *args, **kwargs) -> Image:
        return self.convert(*args, **kwargs)

    def __repr__(self):
        format_string = self.__class__.__name__ + '('
        if self.to_pil.mode is not None:
            format_string += f'mode={self.to_pil.mode}'
        format_string += ')'
        return format_string


class Img2Vec(object):
    """Model wrapper for image embedding"""

    def __init__(
        self, backbone: Union[nn.Module, ScriptModule], trfm: transforms, device: str = 'cpu', 
        func:callable = None, ptrf:callable=None):
        super().__init__()
        self.device = torch.device(device)
        self.backbone = (backbone.eval() if hasattr(backbone, 'eval') else backbone).to(device)
        self.call_backbone = func if func else self.backbone
        self.trfm = trfm
        self.ptrf = ptrf

    def preprocess(self, *xs: Union[Image, np.ndarray]) -> Tensor:
        """
        Transform data before model
        Args:
            *xs: input data

        Returns:
            processed data for model
        """
        return torch.stack([self.trfm(x) for x in xs]).to(self.device)

    @no_grad()
    def forward(self, *xs: Union[Image, np.ndarray]) -> np.ndarray:
        tns = self.preprocess(*xs)
        rts = self.call_backbone(tns)
        rts = self.ptrf(rts) if self.ptrf else rts
        y = rts.cpu().data.numpy()

        return y

    def __call__(self, *args, **kwargs) -> np.ndarray:
        return self.forward(*args, **kwargs)

In [ ]:
vec_trsfm = transforms.Compose([ToPILImage(mode='RGB'),
                                transforms.Resize(size),
                                transforms.CenterCrop(imsz),
                                transforms.ToTensor(),
                                transforms.Normalize(
                                    mean=[0.485, 0.456, 0.406], 
                                    std=[0.229, 0.224, 0.225])])

#### Prepare data

In [ ]:
search_path = path / 'search'

In [ ]:
dir_paths = [dp for dp in search_path.iterdir() if dp.is_dir()]

In [ ]:
dir_paths

In [ ]:
img_pts = [im_pt for dp in dir_paths for im_pt in dp.iterdir() if im_pt.suffix in IMG_SUFF]

In [ ]:
def read_img(im_pt):
    img = cv2.imread(str(im_pt), cv2.IMREAD_ANYCOLOR)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    
    return img

In [ ]:
def read_pil_img(im_pt):
    img = PImage.open(im_pt)
    
    return img

In [ ]:
imgs = [read_img(ip) for ip in img_pts]

In [ ]:
pil_imgs = [read_pil_img(ip) for ip in img_pts]

#### Initialize features extractor

In [ ]:
cut = 1

In [ ]:
body = wide_resnet50_2(pretrained=True)

In [ ]:
body

In [ ]:
class FlattenLayer(nn.Module):
    """Flatten layer"""

    def __init__(self):
        super().__init__()

    def forward(self, x: Tensor) -> Tensor:
        return torch.flatten(x, 1)

In [ ]:
backbone = nn.Sequential(*list(body.children())[:-cut])
net = nn.Sequential(backbone, FlattenLayer())

In [ ]:
net

In [ ]:
img_vec = Img2Vec(net, vec_trsfm, device=device)

In [ ]:
pimgs = tqdm(imgs, desc='Vectorizing images', total=len(imgs), position=0, leave=True)

In [ ]:
vecs = [img_vec(im)[0] for im in pimgs]

In [ ]:
vecs[0].shape, len(vecs)

In [ ]:
img_vecs = list(zip(imgs, vecs))

In [ ]:
vecs[0].shape

In [ ]:
imgs[0].shape[0] * imgs[0].shape[1] * imgs[0].shape[2] 

In [ ]:
img_vecs[0][0].shape

In [ ]:
for im in imgs:
    plt.imshow(im)
    plt.show()
    plt.close()

#### Compare vectors

In [ ]:
from scipy.spatial.distance import cosine

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
def top_vecs(qi, top_k=5, model=img_vec, comp_vecs=img_vecs, normalize:bool=False):
    qv = model(qi)
    qv = qv / qv.norm(dim=-1, keepdim=True) if normalize else qv
    qv = qv[0]
    resul_pts = [(cosine(qv, vc), pt) for pt, vc in comp_vecs]
    resul_pts = sorted(resul_pts, key=lambda x: x[0], reverse=False)
    resul_pts = resul_pts[:top_k]
    
    return resul_pts

#### Query images

- ch_1.jpg
- ch_2.jpg
- ft_1.jpeg
- ft_2.jpg
- rv_1.jpeg
- rv_2.jpeg
- st_1.jpeg
- st_2.jpeg
- st_3.jpg
- ct_1.jpg
- ct_2.jpg

In [ ]:
query_path = path / 'queries'
query_path.mkdir(exist_ok=True)

In [ ]:
qim = read_img(query_path / 'st_2.jpeg')

In [ ]:
res = top_vecs(qim)

In [ ]:
#res

In [ ]:
plt.imshow(qim)
plt.show()
plt.close()

In [ ]:
for dist, res_img in res:
    plt.title(f'dist={dist}')
    plt.imshow(res_img)
    plt.show()
    plt.close()

#### Go into the details

In [ ]:
qim.shape

In [ ]:
plt.imshow(qim)
plt.show()
plt.close()

In [ ]:
qtens = img_vec.preprocess(qim)

In [ ]:
backbone = img_vec.backbone
backbone

In [ ]:
conv_part = backbone[0][:-1]
pool_part = backbone[0][-1]
flat_part = backbone[-1]

In [ ]:
conv_part

In [ ]:
pool_part

In [ ]:
flat_part

In [ ]:
[111111]

In [ ]:
[img1, img2, img3, img4]
[img]

In [ ]:
con_tens = conv_part(qtens)
con_tens[0].shape

In [ ]:
pool_tens = pool_part(con_tens)
pool_tens[0].shape

In [ ]:
lin_tens = flat_part(pool_tens)
lin_tens[0].shape

#### Self-Supervised Learning with SimCLR\n\nIn this section, we will implement a self-supervised learning approach using SimCLR (Simple Framework for Contrastive Learning of Visual Representations). We will train a ResNet18 backbone on the STL-10 dataset (unlabeled split) to learn useful image representations without any labels.\n

##### Data Augmentation\n

In [ ]:
class ContrastiveTransformations:\n    def __init__(self, base_transforms, n_views=2):\n        self.base_transforms = base_transforms\n        self.n_views = n_views\n\n    def __call__(self, x):\n        return [self.base_transforms(x) for _ in range(self.n_views)]\n

In [ ]:
contrast_transforms = transforms.Compose(\n    [\n        transforms.RandomHorizontalFlip(),\n        transforms.RandomResizedCrop(size=96),\n        transforms.RandomApply([transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1)], p=0.8),\n        transforms.RandomGrayscale(p=0.2),\n        transforms.GaussianBlur(kernel_size=9),\n        transforms.ToTensor(),\n        transforms.Normalize((0.5,), (0.5,)),\n    ]\n)\n

##### SimCLR Model\n

In [ ]:
import torchvision\nclass SimCLR(pl.LightningModule):\n    def __init__(self, hidden_dim, lr, temperature, weight_decay, max_epochs=5):\n        super().__init__()\n        self.save_hyperparameters()\n        # Base model f(.)\n        self.convnet = torchvision.models.resnet18(num_classes=4 * hidden_dim)\n        # Projection head g(.)\n        self.convnet.fc = nn.Sequential(\n            self.convnet.fc,  # Linear(ResNet output, 4*hidden_dim)\n            nn.ReLU(inplace=True),\n            nn.Linear(4 * hidden_dim, hidden_dim),\n        )\n\n    def forward(self, x):\n        return self.convnet(x)\n\n    def configure_optimizers(self):\n        optimizer = torch.optim.AdamW(\n            self.parameters(),\n            lr=self.hparams.lr,\n            weight_decay=self.hparams.weight_decay,\n        )\n        lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(\n            optimizer, T_max=self.hparams.max_epochs, eta_min=self.hparams.lr / 50\n        )\n        return [optimizer], [lr_scheduler]\n\n    def info_nce_loss(self, batch, mode="train"):\n        imgs, _ = batch\n        # imgs is a list of [view1, view2]\n        imgs = torch.cat(imgs, dim=0)\n\n        # Encoding and projection\n        feats = self.convnet(imgs)\n        # Calculate cosine similarity\n        cos_sim = F.cosine_similarity(feats[:, None, :], feats[None, :, :], dim=-1)\n        # Mask out cosine similarity to itself\n        self_mask = torch.eye(cos_sim.shape[0], dtype=torch.bool, device=cos_sim.device)\n        cos_sim.masked_fill_(self_mask, -9e15)\n        # Find positive example -> batch_size // 2 away from the original example\n        pos_mask = self_mask.roll(shifts=cos_sim.shape[0] // 2, dims=0)\n        # InfoNCE loss\n        cos_sim = cos_sim / self.hparams.temperature\n        nll = -cos_sim[pos_mask] + torch.logsumexp(cos_sim, dim=-1)\n        nll = nll.mean()\n\n        # Logging loss\n        self.log(mode + "_loss", nll)\n        return nll\n\n    def training_step(self, batch, batch_idx):\n        return self.info_nce_loss(batch, mode="train")\n

##### Training on STL-10\n

In [ ]:
from torch.utils.data import DataLoader\n\ndataset_path = 'data'\nbatch_size = 256\nmax_epochs = 5 # Small number for demonstration\n\nunlabeled_data = datasets.STL10(\n    root=dataset_path, split="unlabeled", download=True, transform=ContrastiveTransformations(contrast_transforms, n_views=2)\n)\ntrain_loader_simclr = DataLoader(\n    unlabeled_data, batch_size=batch_size, shuffle=True, drop_last=True, pin_memory=True, num_workers=0\n)\n

In [ ]:
simclr_model = SimCLR(hidden_dim=128, lr=5e-4, temperature=0.07, weight_decay=1e-4, max_epochs=max_epochs)\ntrainer = pl.Trainer(\n    accelerator="auto",\n    devices=1,\n    max_epochs=max_epochs,\n    callbacks=[\n        LearningRateMonitor("epoch"),\n    ],\n)\ntrainer.fit(simclr_model, train_loader_simclr)\n

#### Search with Jina CLIP v2\n

We will use `jinaai/jina-clip-v2`, a powerful multilingual and multimodal embedding model.\n

In [ ]:
get_ipython().system('pip install transformers timm einops pillow')\n

In [ ]:
from transformers import AutoModel\nfrom numpy.linalg import norm\n\n# Initialize Jina CLIP\njina_clip_model = AutoModel.from_pretrained('jinaai/jina-clip-v2', trust_remote_code=True).to(device)\n

In [ ]:
# Vectorize images with Jina CLIP\nbatches = []\nbatch_size = 32\nfor i in range(0, len(pil_imgs), batch_size):\n    batches.append(pil_imgs[i:i+batch_size])\n\njina_vecs = []\nfor batch in tqdm(batches, desc='Vectorizing with Jina CLIP'):\n    with torch.no_grad():\n        # returns numpy array\n        embeddings = jina_clip_model.encode_image(batch) \n        jina_vecs.append(embeddings)\n\njina_vecs = np.concatenate(jina_vecs, axis=0)\n

In [ ]:
# Re-use top_vecs logic but with jina_vecs\njina_img_vecs = list(zip(imgs, jina_vecs))\n\ndef top_vecs_jina(qim_pil, top_k=5, model=jina_clip_model, comp_vecs=jina_img_vecs):\n    with torch.no_grad():\n        qv = model.encode_image(qim_pil) # Returns (1, D) numpy array\n    \n    qv = qv[0] # (D,)\n    \n    # Cosine distance (scipy cosine is 1 - similarity)\n    resul_pts = [(cosine(qv, vc), pt) for pt, vc in comp_vecs]\n    resul_pts = sorted(resul_pts, key=lambda x: x[0], reverse=False)\n    resul_pts = resul_pts[:top_k]\n    \n    return resul_pts\n

In [ ]:
# Query\nqim_pil = read_pil_img(query_path / 'st_2.jpeg')\nres_jina = top_vecs_jina(qim_pil)\n\nplt.imshow(qim_pil)\nplt.title("Query Image")\nplt.show()\n\nfor dist, res_img in res_jina:\n    plt.title(f'Jina Dist={dist:.4f}')\n    plt.imshow(res_img)\n    plt.show()\n

In [ ]:
plimgs = tqdm(pil_imgs, desc='Vectorizing images (PIL)', total=len(pil_imgs), position=0, leave=True)

- ch_1.jpg
- ch_2.jpg
- ft_1.jpeg
- ft_2.jpg
- rv_1.jpeg
- rv_2.jpeg
- st_1.jpeg
- st_2.jpeg
- st_3.jpg
- ct_1.jpg
- ct_2.jpg

## Text search

``` pip install -U sentence-transformers ```

In [ ]:
from sentence_transformers import SentenceTransformer, util

In [ ]:
model_dbr = SentenceTransformer('paraphrase-distilroberta-base-v1')

In [ ]:
model_dbl = SentenceTransformer('stsb-bert-large')

In [ ]:
model_rbb = SentenceTransformer('stsb-roberta-base')

In [ ]:
model_rbl = SentenceTransformer('stsb-roberta-large')

In [ ]:
#Our sentences we like to encode
sentences = ['This framework generates embeddings for each input sentence',
    'Sentences are passed as a list of string.',
    'The quick brown fox jumps over the lazy dog.']

#Sentences are encoded by calling model.encode()
embeddings = model_dbl.encode(sentences)

#Print the embeddings
for sentence, embedding in zip(sentences, embeddings):
    print("Sentence:", sentence)
    print("Embedding:", embedding)
    print(f'{embedding.shape=}')
    print("")

## Semantic similarity

In [ ]:
# Two lists of sentences
sentences1 = ['The cat sits outside',
             'A man is playing guitar',
             'The new movie is awesome']

sentences2 = ['The dog plays in the garden',
              'A woman watches TV',
              #'The new movie is so great',
              'The new movie is so horrable',
             ]

#Compute embedding for both lists
embeddings1 = model_dbl.encode(sentences1, convert_to_tensor=True)
embeddings2 = model_dbl.encode(sentences2, convert_to_tensor=True)

#Compute cosine-similarits
cosine_scores = util.pytorch_cos_sim(embeddings1, embeddings2)

#Output the pairs with their score
for i in range(len(sentences1)):
    print("{} \t\t {} \t\t Score: {:.4f}".format(sentences1[i], sentences2[i], cosine_scores[i][i]))

## Search

In [ ]:
embedder = SentenceTransformer('paraphrase-distilroberta-base-v1')

In [ ]:
# Corpus with example sentences
corpus = ['A man is eating food.',
          'A man is eating a piece of bread.',
          'The girl is carrying a baby.',
          'A man is riding a horse.',
          'A woman is playing violin.',
          'Two men pushed carts through the woods.',
          'A man is riding a white horse on an enclosed ground.',
          'A monkey is playing drums.',
          'A cheetah is running behind its prey.',
          'A pasta is eating man .',
          'A pasta is eattyen by man .'
          ]
corpus_embeddings = embedder.encode(corpus, convert_to_tensor=True)

# Query sentences:
queries = ['A man is eating pasta.', 'Someone in a gorilla costume is playing a set of drums.', 'A cheetah chases prey on across a field.']


# Find the closest 5 sentences of the corpus for each query sentence based on cosine similarity
top_k = min(5, len(corpus))
for query in queries:
    query_embedding = embedder.encode(query, convert_to_tensor=True)

    # We use cosine-similarity and torch.topk to find the highest 5 scores
    cos_scores = util.pytorch_cos_sim(query_embedding, corpus_embeddings)[0]
    top_results = torch.topk(cos_scores, k=top_k)

    print("\n\n======================\n\n")
    print("Query:", query)
    print("\nTop 5 most similar sentences in corpus:")

    for score, idx in zip(top_results[0], top_results[1]):
        print(corpus[idx], "(Score: {:.4f})".format(score))

    """
    # Alternatively, we can also use util.semantic_search to perform cosine similarty + topk
    hits = util.semantic_search(query_embedding, corpus_embeddings, top_k=5)
    hits = hits[0]      #Get the hits for the first query
    for hit in hits:
        print(corpus[hit['corpus_id']], "(Score: {:.4f})".format(hit['score']))
    """

In [ ]:
embedder

In [ ]:
embedder_body = nn.Sequential(embedder)[-1]

In [ ]:
embedder_body

In [ ]:
sent_vec = embedder_body.encode('The test sentence')

In [ ]:
sent_vec.shape